In [ ]:
# %pip install -r requirements.txt

In [1]:
lazy from sklearn.ensemble import RandomForestClassifier
lazy from sklearn.metrics import accuracy_score, classification_report
lazy from sklearn.model_selection import train_test_split

lazy import pandas as pd
lazy from concurrent.futures import ThreadPoolExecutor
lazy from tqdm import tqdm

lazy from config import RAW_DATASET
lazy from DatasetCleaning.NetworkAnalysis import extract_url_features
lazy from DatasetCleaning.Encoding.encoding import Encoding

In [ ]:
raw_data = pd.read_csv(RAW_DATASET)
raw_data.head(10)

In [ ]:
with ThreadPoolExecutor(max_workers=90) as executor:
    futures = [executor.submit(extract_url_features, url) for url in raw_data['url']] 
    for future in tqdm(futures, total=len(raw_data)):
        raw_data = pd.concat([future.result(), raw_data['type']], axis=1)
# raw_data = pd.concat([raw_data['url'].apply(extract_url_features), raw_data['type']], axis=1)
# print(raw_data.columns)
print(raw_data.columns)

In [ ]:
raw_data.to_csv("dataset/features_extracted.csv", index=False)

In [ ]:
raw_data.head(100)

In [ ]:
X = raw_data.drop(columns=["type"])
y = raw_data["type"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y 
)

print(f"Training data features shape: {X_train.shape}")
print(f"Testing data features shape: {X_test.shape}")

X_train, X_test = Encoding.OneHotenc(Encoding, data = raw_data ,X_train = X_train, X_test=X_test)

In [ ]:
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train,y_train)

In [ ]:
import pickle

model = pickle.load(open('TrainedModels/RFClassifier_13_features.pkl', 'rb'))

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)
print(f'accuracy: {accuracy:.4f}', report, sep='\n')

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y_test, y_pred)

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=model.classes_)
disp.plot()